In [ ]:
# ============================================================
# PCA vs AUTOENCODER vs VARIATIONAL AUTOENCODER ON MNIST
# ============================================================


# ============================================================
# 0. IMPORT PACKAGES
# ============================================================

import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42

BATCH_SIZE = 256
LATENT_DIM = 2

EPOCHS_AE = 20
EPOCHS_VAE = 20

LEARNING_RATE = 1e-3


# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# Use GPU if available, otherwise CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)
print("PyTorch version:", torch.__version__)


# ============================================================
# 2. LOAD MNIST
# ============================================================

print("\nLoading MNIST...")

mnist = fetch_openml(
    "mnist_784",
    version=1,
    as_frame=False
)


# Pixel values are originally between 0 and 255
# Scale them to [0, 1]

X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)


# Standard MNIST split:
#
# First 60,000 observations = training
# Last 10,000 observations  = testing

X_train_flat = X[:60000]
y_train = y[:60000]

X_test_flat = X[60000:]
y_test = y[60000:]


print("Training data shape:", X_train_flat.shape)
print("Test data shape:", X_test_flat.shape)

print("Training labels shape:", y_train.shape)
print("Test labels shape:", y_test.shape)


# ============================================================
# 3. SHOW A FEW ORIGINAL MNIST IMAGES
# ============================================================

fig, axes = plt.subplots(
    2,
    5,
    figsize=(10, 4)
)

for i, ax in enumerate(axes.flat):

    ax.imshow(
        X_train_flat[i].reshape(28, 28),
        cmap="gray"
    )

    ax.set_title(
        f"Label = {y_train[i]}"
    )

    ax.axis("off")


plt.suptitle(
    "Original MNIST Images",
    fontsize=14
)

plt.tight_layout()
plt.show()


# ============================================================
# 4. CREATE PYTORCH DATASETS
# ============================================================

# Reshape:
#
# 784 pixels
#     ->
# 1 x 28 x 28
#
# This is convenient for PyTorch.

X_train_tensor = torch.tensor(
    X_train_flat,
    dtype=torch.float32
).reshape(-1, 1, 28, 28)


X_test_tensor = torch.tensor(
    X_test_flat,
    dtype=torch.float32
).reshape(-1, 1, 28, 28)


y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)


train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("\nPyTorch training observations:",
      len(train_dataset))

print("PyTorch test observations:",
      len(test_dataset))


# ============================================================
# 5. PCA
# ============================================================

print("\n======================================")
print("Training PCA")
print("======================================")


pca = PCA(
    n_components=LATENT_DIM,
    svd_solver="randomized",
    random_state=SEED
)


# ------------------------------------------------------------
# Compress training data
# ------------------------------------------------------------

Z_train_pca = pca.fit_transform(
    X_train_flat
)


# ------------------------------------------------------------
# Compress test data
# ------------------------------------------------------------

Z_test_pca = pca.transform(
    X_test_flat
)


print(
    "Original dimension:",
    X_train_flat.shape[1]
)

print(
    "PCA latent dimension:",
    Z_train_pca.shape[1]
)


# ------------------------------------------------------------
# Reconstruct the test images
# ------------------------------------------------------------

X_test_pca_recon = pca.inverse_transform(
    Z_test_pca
)


# PCA can produce values slightly outside [0, 1]
X_test_pca_recon = np.clip(
    X_test_pca_recon,
    0.0,
    1.0
)


# ------------------------------------------------------------
# PCA reconstruction MSE
# ------------------------------------------------------------

pca_mse = mean_squared_error(
    X_test_flat,
    X_test_pca_recon
)


print(
    "Total explained variance:",
    pca.explained_variance_ratio_.sum()
)

print(
    "PCA Test MSE:",
    pca_mse
)


# ============================================================
# 6. AUTOENCODER
# ============================================================

class Autoencoder(nn.Module):

    def __init__(self, latent_dim=20):

        super().__init__()


        # ----------------------------------------------------
        # ENCODER
        #
        # 784
        #  |
        # 256
        #  |
        # 64
        #  |
        # latent_dim
        # ----------------------------------------------------

        self.encoder = nn.Sequential(

            nn.Flatten(),

            nn.Linear(784, 256),
            nn.ReLU(),

            nn.Linear(256, 64),
            nn.ReLU(),

            nn.Linear(
                64,
                latent_dim
            )
        )


        # ----------------------------------------------------
        # DECODER
        #
        # latent_dim
        #    |
        #   64
        #    |
        #   256
        #    |
        #   784
        # ----------------------------------------------------

        self.decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),
            nn.ReLU(),

            nn.Linear(
                64,
                256
            ),
            nn.ReLU(),

            nn.Linear(
                256,
                784
            )
        )


    def forward(self, x):

        # Encode
        z = self.encoder(x)

        # Decode
        logits = self.decoder(z)

        return logits


# ============================================================
# 7. CREATE AUTOENCODER
# ============================================================

ae = Autoencoder(
    latent_dim=LATENT_DIM
).to(device)


optimizer_ae = torch.optim.Adam(
    ae.parameters(),
    lr=LEARNING_RATE
)


# Since pixel values are between 0 and 1
criterion_ae = nn.BCEWithLogitsLoss()


print("\nAutoencoder:")
print(ae)


# ============================================================
# 8. TRAIN AUTOENCODER
# ============================================================

print("\n======================================")
print("Training Autoencoder")
print("======================================")


ae_losses = []


for epoch in range(EPOCHS_AE):

    ae.train()

    total_loss = 0.0


    for images, _ in train_loader:

        images = images.to(device)


        # Flatten target images
        targets = images.view(
            images.size(0),
            -1
        )


        # --------------------------------
        # Reset gradients
        # --------------------------------

        optimizer_ae.zero_grad()


        # --------------------------------
        # Forward pass
        # --------------------------------

        logits = ae(images)


        # --------------------------------
        # Reconstruction loss
        # --------------------------------

        loss = criterion_ae(
            logits,
            targets
        )


        # --------------------------------
        # Backpropagation
        # --------------------------------

        loss.backward()


        # --------------------------------
        # Update parameters
        # --------------------------------

        optimizer_ae.step()


        total_loss += (
            loss.item()
            * images.size(0)
        )


    average_loss = (
        total_loss
        / len(train_dataset)
    )


    ae_losses.append(
        average_loss
    )


    print(
        f"Epoch "
        f"{epoch + 1:2d}/"
        f"{EPOCHS_AE} "
        f"| Loss = "
        f"{average_loss:.6f}"
    )


# ============================================================
# 9. PLOT AUTOENCODER TRAINING LOSS
# ============================================================

plt.figure(
    figsize=(7, 4)
)

plt.plot(
    range(
        1,
        EPOCHS_AE + 1
    ),
    ae_losses,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")

plt.title(
    "Autoencoder Training Loss"
)

plt.grid(
    alpha=0.3
)

plt.show()


# ============================================================
# 10. AUTOENCODER RECONSTRUCTION
# ============================================================

def reconstruct_ae(
    model,
    loader
):

    model.eval()

    reconstructions = []


    with torch.no_grad():

        for images, _ in loader:

            images = images.to(device)


            # Model output
            logits = model(images)


            # Convert logits to probabilities
            # between 0 and 1
            recon = torch.sigmoid(
                logits
            )


            reconstructions.append(
                recon.cpu().numpy()
            )


    return np.concatenate(
        reconstructions,
        axis=0
    )


X_test_ae_recon = reconstruct_ae(
    ae,
    test_loader
)


print(
    "\nAutoencoder reconstructed shape:",
    X_test_ae_recon.shape
)


# ============================================================
# 11. AUTOENCODER TEST MSE
# ============================================================

ae_mse = mean_squared_error(
    X_test_flat,
    X_test_ae_recon
)


print(
    "Autoencoder Test MSE:",
    ae_mse
)


# ============================================================
# 12. VARIATIONAL AUTOENCODER
# ============================================================

class VAE(nn.Module):

    def __init__(
        self,
        latent_dim=20
    ):

        super().__init__()


        # ----------------------------------------------------
        # ENCODER
        # ----------------------------------------------------

        self.encoder = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                784,
                256
            ),
            nn.ReLU(),

            nn.Linear(
                256,
                64
            ),
            nn.ReLU()
        )


        # ----------------------------------------------------
        # Mean of latent distribution
        # ----------------------------------------------------

        self.fc_mu = nn.Linear(
            64,
            latent_dim
        )


        # ----------------------------------------------------
        # log variance of latent distribution
        # ----------------------------------------------------

        self.fc_logvar = nn.Linear(
            64,
            latent_dim
        )


        # ----------------------------------------------------
        # DECODER
        # ----------------------------------------------------

        self.decoder = nn.Sequential(

            nn.Linear(
                latent_dim,
                64
            ),
            nn.ReLU(),

            nn.Linear(
                64,
                256
            ),
            nn.ReLU(),

            nn.Linear(
                256,
                784
            )
        )


    # ========================================================
    # Encoder
    # ========================================================

    def encode(self, x):

        h = self.encoder(x)

        mu = self.fc_mu(h)

        logvar = self.fc_logvar(h)

        return mu, logvar


    # ========================================================
    # Reparameterization trick
    #
    # z = mu + sigma * epsilon
    #
    # epsilon ~ N(0, I)
    # ========================================================

    def reparameterize(
        self,
        mu,
        logvar
    ):

        # logvar = log(sigma^2)
        #
        # therefore
        #
        # sigma = exp(0.5 * logvar)

        std = torch.exp(
            0.5 * logvar
        )


        epsilon = torch.randn_like(
            std
        )


        z = (
            mu
            + std * epsilon
        )


        return z


    # ========================================================
    # Decoder
    # ========================================================

    def decode(
        self,
        z
    ):

        return self.decoder(z)


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        x
    ):

        # Get latent distribution
        mu, logvar = self.encode(x)


        # Sample z
        z = self.reparameterize(
            mu,
            logvar
        )


        # Decode
        logits = self.decode(z)


        return (
            logits,
            mu,
            logvar
        )


# ============================================================
# 13. CREATE VAE
# ============================================================

vae = VAE(
    latent_dim=LATENT_DIM
).to(device)


optimizer_vae = torch.optim.Adam(
    vae.parameters(),
    lr=LEARNING_RATE
)


print("\nVariational Autoencoder:")
print(vae)


# ============================================================
# 14. VAE LOSS FUNCTION
# ============================================================

def vae_loss(
    logits,
    targets,
    mu,
    logvar
):

    # --------------------------------------------------------
    # 1. Reconstruction loss
    # --------------------------------------------------------

    reconstruction_loss = (
        nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="sum"
        )
    )


    # --------------------------------------------------------
    # 2. KL divergence
    #
    # KL(
    # q(z|x)
    # ||
    # N(0,I)
    # )
    # --------------------------------------------------------

    kl_loss = -0.5 * torch.sum(

        1
        + logvar
        - mu.pow(2)
        - logvar.exp()

    )


    # --------------------------------------------------------
    # Total VAE loss
    # --------------------------------------------------------

    total_loss = (
        reconstruction_loss
        + kl_loss
    )


    return (
        total_loss,
        reconstruction_loss,
        kl_loss
    )


# ============================================================
# 15. TRAIN VAE
# ============================================================

print("\n======================================")
print("Training Variational Autoencoder")
print("======================================")


vae_losses = []

vae_recon_losses = []

vae_kl_losses = []


for epoch in range(EPOCHS_VAE):

    vae.train()


    total_loss = 0.0

    total_recon = 0.0

    total_kl = 0.0


    for images, _ in train_loader:

        images = images.to(device)


        targets = images.view(
            images.size(0),
            -1
        )


        # --------------------------------
        # Reset gradients
        # --------------------------------

        optimizer_vae.zero_grad()


        # --------------------------------
        # Forward pass
        # --------------------------------

        logits, mu, logvar = vae(
            images
        )


        # --------------------------------
        # Loss
        # --------------------------------

        loss, recon_loss, kl_loss = vae_loss(

            logits,
            targets,
            mu,
            logvar

        )


        # --------------------------------
        # Backpropagation
        # --------------------------------

        loss.backward()


        # --------------------------------
        # Update parameters
        # --------------------------------

        optimizer_vae.step()


        total_loss += loss.item()

        total_recon += (
            recon_loss.item()
        )

        total_kl += (
            kl_loss.item()
        )


    # Loss per observation
    average_loss = (
        total_loss
        / len(train_dataset)
    )


    average_recon = (
        total_recon
        / len(train_dataset)
    )


    average_kl = (
        total_kl
        / len(train_dataset)
    )


    vae_losses.append(
        average_loss
    )

    vae_recon_losses.append(
        average_recon
    )

    vae_kl_losses.append(
        average_kl
    )


    print(
        f"Epoch "
        f"{epoch + 1:2d}/"
        f"{EPOCHS_VAE} "
        f"| Total = "
        f"{average_loss:.3f} "
        f"| Recon = "
        f"{average_recon:.3f} "
        f"| KL = "
        f"{average_kl:.3f}"
    )


# ============================================================
# 16. PLOT VAE TRAINING LOSS
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    range(
        1,
        EPOCHS_VAE + 1
    ),
    vae_losses,
    marker="o",
    label="Total"
)


plt.plot(
    range(
        1,
        EPOCHS_VAE + 1
    ),
    vae_recon_losses,
    marker="o",
    label="Reconstruction"
)


plt.plot(
    range(
        1,
        EPOCHS_VAE + 1
    ),
    vae_kl_losses,
    marker="o",
    label="KL"
)


plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "VAE Training Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()


# ============================================================
# 17. VAE RECONSTRUCTION
# ============================================================

def reconstruct_vae(
    model,
    loader
):

    model.eval()

    reconstructions = []


    with torch.no_grad():

        for images, _ in loader:

            images = images.to(device)


            # --------------------------------
            # Encoder produces
            #
            # mu
            # log variance
            # --------------------------------

            mu, logvar = model.encode(
                images
            )


            # --------------------------------
            # For reconstruction,
            # use mu instead of random z.
            #
            # This gives deterministic
            # reconstructions.
            # --------------------------------

            logits = model.decode(
                mu
            )


            recon = torch.sigmoid(
                logits
            )


            reconstructions.append(
                recon.cpu().numpy()
            )


    return np.concatenate(
        reconstructions,
        axis=0
    )


X_test_vae_recon = reconstruct_vae(
    vae,
    test_loader
)


print(
    "\nVAE reconstructed shape:",
    X_test_vae_recon.shape
)


# ============================================================
# 18. VAE TEST MSE
# ============================================================

vae_mse = mean_squared_error(
    X_test_flat,
    X_test_vae_recon
)


print(
    "VAE Test MSE:",
    vae_mse
)


# ============================================================
# 19. FINAL NUMERICAL COMPARISON
# ============================================================

print("\n")
print("=" * 50)

print(
    "FINAL RECONSTRUCTION COMPARISON"
)

print("=" * 50)


print(
    f"PCA MSE          = "
    f"{pca_mse:.6f}"
)

print(
    f"Autoencoder MSE  = "
    f"{ae_mse:.6f}"
)

print(
    f"VAE MSE          = "
    f"{vae_mse:.6f}"
)


print("=" * 50)


# ============================================================
# 20. COMPARE RECONSTRUCTED IMAGES
# ============================================================

N_IMAGES = 10


# Select random test images
rng = np.random.default_rng(
    SEED
)

indices = rng.choice(
    len(X_test_flat),
    N_IMAGES,
    replace=False
)


fig, axes = plt.subplots(
    4,
    N_IMAGES,
    figsize=(16, 7)
)


row_names = [
    "Original",
    "PCA",
    "Autoencoder",
    "VAE"
]


for j, idx in enumerate(indices):


    # --------------------------------------------------------
    # Original
    # --------------------------------------------------------

    original = (
        X_test_flat[idx]
        .reshape(28, 28)
    )


    # --------------------------------------------------------
    # PCA
    # --------------------------------------------------------

    pca_image = (
        X_test_pca_recon[idx]
        .reshape(28, 28)
    )


    # --------------------------------------------------------
    # Autoencoder
    # --------------------------------------------------------

    ae_image = (
        X_test_ae_recon[idx]
        .reshape(28, 28)
    )


    # --------------------------------------------------------
    # VAE
    # --------------------------------------------------------

    vae_image = (
        X_test_vae_recon[idx]
        .reshape(28, 28)
    )


    images = [
        original,
        pca_image,
        ae_image,
        vae_image
    ]


    for i in range(4):

        axes[i, j].imshow(
            images[i],
            cmap="gray",
            vmin=0,
            vmax=1
        )


        axes[i, j].axis("off")


        # Digit label above original image
        if i == 0:

            axes[i, j].set_title(
                f"{y_test[idx]}"
            )


        # Row names
        if j == 0:

            axes[i, j].text(
                -0.12, 0.5,
                row_names[i],
                transform=axes[i, j].transAxes,
                fontsize=12,
                ha="right",
                va="center"
            )


plt.suptitle(
    f"MNIST Reconstruction Comparison "
    f"(Latent Dimension = {LATENT_DIM})",
    fontsize=16
)


plt.tight_layout(rect=[0.18, 0, 1, 0.95])


#plt.savefig("mnist_reconstruction_comparison.png", dpi=200, bbox_inches="tight")


plt.show()


# ============================================================
# 21. MSE BAR PLOT
# ============================================================

methods = [
    "PCA",
    "Autoencoder",
    "VAE"
]


mse_values = [
    pca_mse,
    ae_mse,
    vae_mse
]


plt.figure(
    figsize=(7, 5)
)


bars = plt.bar(
    methods,
    mse_values
)


plt.ylabel(
    "Test Reconstruction MSE"
)


plt.title(
    f"MNIST Reconstruction Error\n"
    f"Latent Dimension = {LATENT_DIM}"
)


# Add numbers above bars
for bar, value in zip(
    bars,
    mse_values
):

    plt.text(

        bar.get_x()
        + bar.get_width() / 2,

        bar.get_height(),

        f"{value:.4f}",

        ha="center",
        va="bottom",

        fontsize=11
    )


plt.tight_layout()


#plt.savefig("mnist_mse_comparison.png", dpi=200,bbox_inches="tight")


plt.show()


# ============================================================
# 22. PCA COMPONENTS
# ============================================================

# Show first 10 principal component directions
# as images.

fig, axes = plt.subplots(
    2,
    5,
    figsize=(10, 4)
)


for i, ax in enumerate(
    axes.flat
):

    component = (
        pca.components_[i]
        .reshape(28, 28)
    )


    ax.imshow(
        component,
        cmap="gray"
    )


    ax.set_title(
        f"PC {i + 1}"
    )


    ax.axis("off")


plt.suptitle(
    "First 10 PCA Components",
    fontsize=14
)

plt.tight_layout()

plt.show()


# ============================================================
# 23. GENERATE NEW DIGITS WITH THE VAE
# ============================================================

vae.eval()


N_GENERATED = 20


with torch.no_grad():

    # --------------------------------------------------------
    # Sample z from
    #
    # N(0, I)
    # --------------------------------------------------------

    z = torch.randn(
        N_GENERATED,
        LATENT_DIM,
        device=device
    )


    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    generated_logits = vae.decode(
        z
    )


    generated = torch.sigmoid(
        generated_logits
    )


    generated = (
        generated
        .cpu()
        .numpy()
    )


# ============================================================
# 24. SHOW GENERATED DIGITS
# ============================================================

fig, axes = plt.subplots(
    2,
    10,
    figsize=(14, 3)
)


for i, ax in enumerate(
    axes.flat
):

    ax.imshow(
        generated[i].reshape(
            28,
            28
        ),
        cmap="gray",
        vmin=0,
        vmax=1
    )


    ax.axis("off")


plt.suptitle(
    "New Digits Generated by the Variational Autoencoder",
    fontsize=15
)


plt.tight_layout()


#plt.savefig("vae_generated_digits.png", dpi=200, bbox_inches="tight")


plt.show()


# ============================================================
# 25. SIMPLE SUMMARY
# ============================================================

print("\n")
print("=" * 60)

print("SUMMARY")

print("=" * 60)

print(
    "Original dimension:",
    784
)

print(
    "Latent dimension:",
    LATENT_DIM
)

print()

print(
    f"PCA reconstruction MSE:         "
    f"{pca_mse:.6f}"
)

print(
    f"Autoencoder reconstruction MSE: "
    f"{ae_mse:.6f}"
)

print(
    f"VAE reconstruction MSE:         "
    f"{vae_mse:.6f}"
)

print()

print(
    "PCA: linear dimension reduction"
)

print(
    "AE: nonlinear dimension reduction"
)

print(
    "VAE: nonlinear + probabilistic dimension reduction"
)

print(
    "VAE can also generate new images."
)

print("=" * 60)

Using device: cpu
PyTorch version: 2.12.0+cpu

Loading MNIST...
